### REINFORCE 控制 CartPole

[**Code**](./reinforce_cartpole.py)

1. 策略网络 (`:40-72`)

```
4（状态）→ 128 → 128 → 2 (动作) → softmax
```

输出概率分布而非动作，这就是策略梯度与 DQN 分家的地方——DQN 输出 `Q(s,a)` 再取 argmax，REINFORCE 直接对 `π(a|s)` 求导。

2. 折扣回报 `compute_returns`（`:78-103`）

```python
for reward in reversed(rewards):
    G = reward + gamma * G
    returns.insert(0, G)
```

倒序递推 `G_t = r_t + γ·G_{t+1}`，O(T) 一趟算完。CartPole 奖励恒为 +1，所以 `G_0 ≈ (1−0.99^T)/0.01`，回合越长权重越大——这条决定了后面所有的方差问题。

3. 轨迹采集 `collect_episode`（`:109-156`）

```python
with torch.no_grad():          # 采集阶段不建图
    probs = policy(state_tensor)
action = torch.distributions.Categorical(probs).sample()   # 按概率采样，不是 argmax
```

- 采样而非取最大：探索机制就在这一行。第一章 PPO 用 `Categorical(logits)`，第四章 DQN 用 ε-greedy，这里是策略内生探索。
- on-policy 铁律（`:114-116` 注释）：本轮数据只服务本轮更新，用完即弃，下轮重采。

4. 核心更新 `train_one_episode`（`:162-206`）★ 本章最重要的一段

```
action_probs = probs.gather(1, actions.unsqueeze(1)).squeeze(1)  # π(a_t|s_t)
log_probs    = torch.log(action_probs + 1e-8)
loss         = -(log_probs * returns_tensor).mean()
```

数学上 `∇loss = −∇J(θ)`，所以最小化 loss 等价于最大化期望回报。
直观解释（`:195-198` 的注释说得很准）：
- `G_t > 0`（这一步走得好）→ `loss < 0` → 梯度下降把 `log π` 推高 → 该动作概率变大
- `G_t < 0` → 反向压低
这就是"好动作多做、坏动作少做"的数学落地。

三个工程细节：

|代码|作用|
|---|---|
|`torch.no_grad()` 在采集阶段（`:138`）|省算力，采集不需要梯度|
|`gather`（`:192`）|从整行概率里挑出实际执行动作那一个|
|`+ 1e-8`（`:193`）|防 `log(0)`|

5. 训练主循环（:252-275）

```
500 个 episode，每个 episode：
  collect_episode → compute_returns → train_one_episode
  每 50 回合打印 last-50 平均奖励
```

一个回合一次梯度更新——这是 REINFORCE 样本效率低的根源（对比 PPO 用 2048 步、PPO 一轮更新 10 个 epoch）。

6. 关键缺陷：没有基线
`returns` 直接当权重。CartPole 的 `G_t` 量级在 1~100 之间，单回合的梯度方向几乎被噪声主导。这是下一节要解决的问题。

### 价值基线控制 CartPole

[**Code**](./reinforce_with_baseline.py)

结构与前者完全对齐，只改了三处。

》改动 1：新增 ValueNetwork（`:38-50`）

```
4 (状态) → 128 → 128 → 1 (标量 V(s)) → squeeze
```

输出层是 `1` 而非 `2` —— 回归量，不是分类。

》改动 2：权重换成优势 `A = G − V(s)`（`:135-147`）

```python
# 先训价值网络
value_loss = MSE(value_net(states_t), returns_t)
value_optimizer.step()

# 再用优势更新策略
advantages = returns_t - value_net(states_t)      # :142
policy_loss = -(log_probs * advantages).mean()    # :147
```

为什么有效：理论保证 `E[G_t − b(s_t)] = E[G_t]`（基线不依赖动作时，期望不变），但 `Var(G − V) ≪ Var(G)`。梯度期望没变，噪声小了，学习自然稳。

对照 Chapter 3 的 `Q[a] += (r − Q[a])/count` —— 同一个减法思想，只不过基线从查表升级成了神经网络回归。

》改动 3：渲染 GIF（`:158-177`）

```python
action = int(torch.argmax(policy(state_t), dim=-1).item())   # :168
```

训练用采样，演示用 argmax——把探索噪声关掉，展示策略的确定性最优行为。

```python
frames.append(env.render())        # 先渲染再走一步，保证首帧完整
imageio.mimsave(output, frames, fps=30)
```

主流程（`:193-205`）：训两个策略 → 各渲一个 GIF → 打印对比。训练用 `seed=args.seed`，baseline 用 `seed+100`。

实测结果：
```
Vanilla  final-50 avg:  75.08 | GIF return 117  (118 帧)
Baseline final-50 avg: 477.04  | GIF return 500  (501 帧，顶到上限)
```

|实验|更新信号|额外网络|直观含义|
|---|---|---|---|
|Vanilla REINFORCE|`G_t`|无|只看这一回合之后实际拿了多少分|
|REINFORCE + Value Baseline|`G_t - V(s_t)`|Value Network|看实际结果比当前状态的平均预期好多少|

---

|两脚本对照 | `reinforce_cartpole.py` | `render_cartpole_baseline.py` |
|---|---|---| 
| 网络 | PolicyNet（输出概率） | PolicyNet + **ValueNet**（标量） |
| 优化器 | 1 个 Adam | **2 个 Adam**，各自反向传播 | 
| 权重 | `G_t` | `A_t = G_t − V(s_t)` | 
| 回合更新次数 | 1 次 | 1 次（价值 + 策略各1 次） | 
| 产物 | 训练曲线 PNG | **两个 GIF** + 分数对比 |